In [ ]:
import os
import time
import json
import random
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models


# =========================
# 설정
# =========================
DATA_ROOT = r"C:\Users\Konyang\Desktop\cropped_preprocessed_dataset"
SAVE_DIR = r"C:\Users\Konyang\Desktop\new_conv"
MODEL_SAVE_DIR = os.path.join(SAVE_DIR, "epoch_models")

BATCH_SIZE = 8
NUM_EPOCHS = 25
LR = 1e-4
IMG_SIZE = 320
NUM_WORKERS = 0
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================
# 시드 고정
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =========================
# 정확도 계산
# =========================
def calc_accuracy(outputs, labels):
    _, preds = torch.max(outputs, 1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct / total


# =========================
# 한 epoch 학습
# =========================
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    running_loss = 0.0
    running_acc = 0.0
    total_samples = 0

    pbar = tqdm(loader, desc="Train", leave=False)

    for images, labels in pbar:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        acc = calc_accuracy(outputs, labels)

        running_loss += loss.item() * batch_size
        running_acc += acc * batch_size
        total_samples += batch_size

        pbar.set_postfix({
            "loss": f"{running_loss / total_samples:.4f}",
            "acc": f"{running_acc / total_samples:.4f}"
        })

    epoch_loss = running_loss / total_samples
    epoch_acc = running_acc / total_samples
    return epoch_loss, epoch_acc


# =========================
# 평가
# =========================
@torch.no_grad()
def evaluate(model, loader, criterion, desc="Val"):
    model.eval()

    running_loss = 0.0
    running_acc = 0.0
    total_samples = 0

    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc=desc, leave=False)

    for images, labels in pbar:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        _, preds = torch.max(outputs, 1)
        acc = calc_accuracy(outputs, labels)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        running_acc += acc * batch_size
        total_samples += batch_size

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

        pbar.set_postfix({
            "loss": f"{running_loss / total_samples:.4f}",
            "acc": f"{running_acc / total_samples:.4f}"
        })

    epoch_loss = running_loss / total_samples
    epoch_acc = running_acc / total_samples
    return epoch_loss, epoch_acc, all_labels, all_preds


def main():
    set_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    os.makedirs(SAVE_DIR, exist_ok=True)
    os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

    train_dir = os.path.join(DATA_ROOT, "train")
    val_dir = os.path.join(DATA_ROOT, "val")
    test_dir = os.path.join(DATA_ROOT, "test")

    assert os.path.exists(train_dir), f"train 폴더 없음: {train_dir}"
    assert os.path.exists(val_dir), f"val 폴더 없음: {val_dir}"
    assert os.path.exists(test_dir), f"test 폴더 없음: {test_dir}"

    print("=" * 70)
    print("데이터 경로 확인")
    print("=" * 70)
    print("DATA_ROOT      :", DATA_ROOT)
    print("SAVE_DIR       :", SAVE_DIR)
    print("MODEL_SAVE_DIR :", MODEL_SAVE_DIR)
    print("DEVICE         :", DEVICE)
    print("=" * 70)

    # =========================
    # Transform
    # =========================
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    # =========================
    # Dataset / Loader
    # =========================
    train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
    val_dataset = datasets.ImageFolder(val_dir, transform=eval_transform)
    test_dataset = datasets.ImageFolder(test_dir, transform=eval_transform)

    class_names = train_dataset.classes
    num_classes = len(class_names)

    print("=" * 70)
    print("클래스 정보")
    print("=" * 70)
    print("class_to_idx:", train_dataset.class_to_idx)
    print("num_classes :", num_classes)
    print("train 개수  :", len(train_dataset))
    print("val 개수    :", len(val_dataset))
    print("test 개수   :", len(test_dataset))
    print("=" * 70)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    # =========================
    # 모델: ConvNeXt-Tiny
    # =========================
    print("ConvNeXt-Tiny 불러오는 중...")

    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

    # ConvNeXt-Tiny classifier 구조:
    # model.classifier[2]가 마지막 Linear 층
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)

    model = model.to(DEVICE)

    # =========================
    # Loss / Optimizer / Scheduler
    # =========================
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )

    # =========================
    # 학습 기록
    # =========================
    best_val_acc = 0.0
    best_epoch_path = ""

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    start_time = time.time()

    print("\n" + "=" * 70)
    print("학습 시작")
    print("=" * 70)

    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch [{epoch + 1}/{NUM_EPOCHS}]")

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, desc="Val")

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
        print(f"LR        : {optimizer.param_groups[0]['lr']:.6f}")

        # epoch마다 모델 개별 저장
        epoch_save_name = (
            f"epoch_{epoch + 1:02d}"
            f"_trainLoss_{train_loss:.4f}"
            f"_trainAcc_{train_acc:.4f}"
            f"_valLoss_{val_loss:.4f}"
            f"_valAcc_{val_acc:.4f}.pth"
        )

        epoch_save_path = os.path.join(MODEL_SAVE_DIR, epoch_save_name)

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "class_names": class_names,
            "class_to_idx": train_dataset.class_to_idx
        }, epoch_save_path)

        print(f">>> epoch 모델 저장 완료: {epoch_save_name}")

        # history 매 epoch 저장
        with open(os.path.join(SAVE_DIR, "history.json"), "w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=2)

        # 최고 val acc 추적
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch_path = epoch_save_path

    total_time = time.time() - start_time

    print("\n" + "=" * 70)
    print("학습 완료")
    print("=" * 70)
    print(f"총 소요 시간 : {total_time / 60:.2f}분")
    print(f"Best Val Acc : {best_val_acc:.4f}")
    print(f"Best Model   : {best_epoch_path}")
    print("=" * 70)

    # =========================
    # 최고 성능 모델 다시 로드
    # =========================
    if best_epoch_path:
        best_checkpoint = torch.load(best_epoch_path, map_location=DEVICE)
        model.load_state_dict(best_checkpoint["model_state_dict"])

    # =========================
    # 테스트
    # =========================
    print("\n테스트 평가 시작")
    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, desc="Test")

    report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    cm = confusion_matrix(y_true, y_pred)

    print(f"\nTest Loss: {test_loss:.4f}")
    print(f"Test Acc : {test_acc:.4f}")
    print("\nClassification Report")
    print(report)
    print("\nConfusion Matrix")
    print(cm)

    # =========================
    # 결과 저장
    # =========================
    with open(os.path.join(SAVE_DIR, "test_report.txt"), "w", encoding="utf-8") as f:
        f.write(f"Best Epoch Model: {best_epoch_path}\n")
        f.write(f"Best Val Acc: {best_val_acc:.4f}\n")
        f.write(f"Test Loss: {test_loss:.4f}\n")
        f.write(f"Test Acc : {test_acc:.4f}\n\n")
        f.write("Classes\n")
        f.write(str(class_names))
        f.write("\n\nClassification Report\n")
        f.write(report)
        f.write("\n\nConfusion Matrix\n")
        f.write(np.array2string(cm))

    with open(os.path.join(SAVE_DIR, "class_mapping.json"), "w", encoding="utf-8") as f:
        json.dump(train_dataset.class_to_idx, f, ensure_ascii=False, indent=2)

    # Loss 그래프 저장
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 6))
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss Curve")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "loss_curve.png"))
    plt.close()

    # Accuracy 그래프 저장
    plt.figure(figsize=(8, 6))
    plt.plot(epochs, history["train_acc"], label="Train Acc")
    plt.plot(epochs, history["val_acc"], label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy Curve")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "acc_curve.png"))
    plt.close()

    print("\n결과 저장 완료")
    print(os.path.join(SAVE_DIR, "history.json"))
    print(os.path.join(SAVE_DIR, "test_report.txt"))
    print(os.path.join(SAVE_DIR, "class_mapping.json"))
    print(os.path.join(SAVE_DIR, "loss_curve.png"))
    print(os.path.join(SAVE_DIR, "acc_curve.png"))
    print(MODEL_SAVE_DIR)


if __name__ == "__main__":
    main()